# LAB | Introduction to MLOps

Answer the questions,

Data source: https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [11]:
!pip install pyarrow scikit-learn

In [12]:
df_jan = pd.read_parquet('./data/fhv_tripdata_2021-01.parquet')
df_feb = pd.read_parquet('./data/fhv_tripdata_2021-02.parquet')

df_jan.head()

,dispatching_base_num,pickup_datetime,dropOff_datetime,PUlocationID,DOlocationID,SR_Flag,Affiliated_base_number
0,B00009,2021-01-01 00:27:00,2021-01-01 00:44:00,NaN,NaN,None,B00009
1,B00009,2021-01-01 00:50:00,2021-01-01 01:07:00,NaN,NaN,None,B00009
2,B00013,2021-01-01 00:01:00,2021-01-01 01:51:00,NaN,NaN,None,B00013
3,B00037,2021-01-01 00:13:09,2021-01-01 00:21:26,NaN,72.0,None,B00037
4,B00037,2021-01-01 00:38:31,2021-01-01 00:53:44,NaN,61.0,None,B00037


**Q1: Read the data for January. How many records are there?**

In [13]:
# Q1: How many records are there in January?
print(f"Number of records in January: {len(df_jan)}")

Number of records in January: 1154112


**Q2: What's the average trip duration in January?**

In [14]:
# Q2: What's the average trip duration in January?

# Compute trip duration in minutes
df_jan['duration'] = (df_jan['dropOff_datetime'] - df_jan['pickup_datetime']).dt.total_seconds() / 60

print(f"Average trip duration in January: {df_jan['duration'].mean():.2f} minutes")

Average trip duration in January: 19.17 minutes


**Q3: How many records did you drop?**

In [15]:
# Q3: How many records did you drop?
# Keep only trips between 1 and 60 minutes (remove outliers)

records_before = len(df_jan)
df_jan = df_jan[(df_jan['duration'] >= 1) & (df_jan['duration'] <= 60)]
records_after = len(df_jan)

print(f"Records before filtering: {records_before}")
print(f"Records after filtering: {records_after}")
print(f"Records dropped: {records_before - records_after}")
print(f"Fraction dropped: {(records_before - records_after) / records_before:.2%}")

Records before filtering: 1154112
Records after filtering: 1109826
Records dropped: 44286
Fraction dropped: 3.84%


**What's the fractions of missing values for the pickup location ID? I.e. fraction of "-1"s after you filled the NAs.**

In [16]:
# Q4: What's the fraction of missing values for the pickup location ID?

# Check missing values before filling
missing_before = df_jan['PUlocationID'].isna().sum()
print(f"Missing PUlocationID values: {missing_before}")

# Fill NAs with -1
df_jan['PUlocationID'] = df_jan['PUlocationID'].fillna(-1)
df_jan['DOlocationID'] = df_jan['DOlocationID'].fillna(-1)

# Fraction of -1s in PUlocationID
fraction_missing = (df_jan['PUlocationID'] == -1).mean()
print(f"Fraction of -1s in PUlocationID: {fraction_missing:.2%}")

Missing PUlocationID values: 927008
Fraction of -1s in PUlocationID: 83.53%


**Q5: What's the dimensionality of this matrix? (The number of columns).**

In [17]:
# Q5: What's the dimensionality of the feature matrix (number of columns)?

# Convert location IDs to strings for one-hot encoding
df_jan['PUlocationID'] = df_jan['PUlocationID'].astype(str)
df_jan['DOlocationID'] = df_jan['DOlocationID'].astype(str)

# Create feature dictionaries
categorical = ['PUlocationID', 'DOlocationID']
train_dicts = df_jan[categorical].to_dict(orient='records')

# Fit DictVectorizer
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

print(f"Dimensionality of the feature matrix: {X_train.shape[1]} columns")
print(f"Shape: {X_train.shape}")

Dimensionality of the feature matrix: 525 columns
Shape: (1109826, 525)


**Q6: What's the RMSE on train?**

In [18]:
# Q6: What's the RMSE on train?

y_train = df_jan['duration'].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
rmse_train = root_mean_squared_error(y_train, y_pred_train)

print(f"RMSE on training data: {rmse_train:.2f}")

RMSE on training data: 10.53


Now, let's put data preprocssing steps in a function so that we can process the validation set in the same way as well.

In [19]:
# Preprocessing function to reuse on validation data

def preprocess(df, dv=None, fit_dv=False):
    # Compute trip duration in minutes
    df['duration'] = (df['dropOff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60
    
    # Filter outliers: keep only trips between 1 and 60 minutes
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)].copy()
    
    # Fill missing location IDs with -1
    df['PUlocationID'] = df['PUlocationID'].fillna(-1).astype(str)
    df['DOlocationID'] = df['DOlocationID'].fillna(-1).astype(str)
    
    # One-hot encode categorical features
    categorical = ['PUlocationID', 'DOlocationID']
    dicts = df[categorical].to_dict(orient='records')
    
    if fit_dv:
        dv = DictVectorizer()
        X = dv.fit_transform(dicts)
    else:
        X = dv.transform(dicts)
    
    y = df['duration'].values
    
    return X, y, dv

**Q7: What's the RMSE on validation?**

In [20]:
# Q7: What's the RMSE on validation?

# Process validation data (February) using the same DictVectorizer fitted on training data
X_val, y_val, _ = preprocess(df_feb, dv=dv, fit_dv=False)

y_pred_val = lr.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred_val)

print(f"RMSE on validation data: {rmse_val:.2f}")

RMSE on validation data: 11.01


## Why to use MLOps as we learn from this excercise ##

This exercise demonstrates several reasons why MLOps is essential:

1. **Reproducibility**: We had to manually track preprocessing steps (duration calculation, outlier filtering, missing value imputation, one-hot encoding). MLOps tools help automate and version these pipelines so they are consistently applied across training and validation data.

2. **Data & Model Versioning**: We used January data for training and February for validation. In production, new data arrives continuously. MLOps provides systematic tracking of which data and model versions were used, making it easy to reproduce or roll back results.

3. **Pipeline Automation**: We had to manually rewrite the same preprocessing logic into a reusable function. MLOps frameworks (like MLflow, Kubeflow, or ZenML) allow you to define these steps as automated, reusable pipeline components.

4. **Experiment Tracking**: We computed RMSE on train and validation sets, but without MLOps tooling these metrics are lost once the notebook session ends. Tools like MLflow track all experiments, hyperparameters, and metrics persistently.

5. **Deployment & Monitoring**: Moving from a notebook to a production-ready model serving API requires packaging, deployment, and monitoring for data drift and model degradation — all of which MLOps addresses.

In short, MLOps bridges the gap between experimentation in notebooks and reliable, scalable machine learning in production.

## BONUS: 

Now, try and run this notebook on AWS Instance